# 7. MCH clustering

Part of the **[Fig. 3 chapter](fig3.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
from glob import glob

import anndata
import matplotlib as mpl
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
import seaborn as sns

from ALLCools.clustering import *
from ALLCools.integration.seurat_class import SeuratIntegration
from ALLCools.plot import *
from ALLCools.mcds import MCDS

from sklearn.metrics import pairwise_distances, roc_auc_score, average_precision_score
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

mpl.style.use("default")
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = "Helvetica"

import warnings
warnings.filterwarnings("ignore")


In [ ]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [ ]:
indir = f'{ENTEX_ROOT}/'
outdir = f'{indir}analysis/mCH_clustering/'


In [ ]:
meta = anndata.read_h5ad(f'{indir}clustering/merged/5kCG100k3C_summary.h5ad').obs


In [ ]:
meta['L1'] = meta['L1'].astype(str)
meta.loc[meta['L1']=='c7', 'L1'] = 'c35'
meta.loc[(meta['L1']=='c35') & meta['L2_any'].isin(['c0','c10','c11']), 'L1'] = 'c36'


In [ ]:
cluster_meta = pd.read_csv(f'{indir}clustering/merged/L2final_celltype_L2both_new.tsv', sep='\t', index_col=None, header=0)
clustermap = cluster_meta[['celltype_L2_both_abbr', 'L2_final']].drop_duplicates().set_index('L2_final')['celltype_L2_both_abbr']
meta['celltype_L2_both_abbr'] = meta['L2_final'].astype(str).map(clustermap)


In [ ]:
gene_meta_path = f'{REF_ROOT}/hg38/gencode/v30/gencode.v30.annotation.gene.flat.tsv.gz'
black_list_path = f'{REF_ROOT}/blacklist/hg38-blacklist.v2.bed.gz'

chrom_to_remove = ['chrL', 'chrM', 'chrX', 'chrY']


In [ ]:
mcds_path_list = np.sort(glob(f'{indir}mcds/*.mcds'))
print(len(mcds_path_list))

In [ ]:
obs_dim = 'cell'
var_dim = 'chrom100k'


In [ ]:
mcds = MCDS.open(mcds_path_list, var_dim=var_dim)
mcds

In [ ]:
mcds = mcds.sel({obs_dim: mcds.get_index('cell').intersection(meta.index)})
mcds = mcds.remove_chromosome(exclude_chromosome=chrom_to_remove, var_dim=var_dim)
mcds = mcds.remove_black_list_region(black_list_path=black_list_path, f=0.5)


In [ ]:
mcds


In [ ]:
# global_cg = mcds[f'{var_dim}_da'].sel(mc_type='CGN').sum(dim=var_dim).to_pandas()
# global_ch = mcds[f'{var_dim}_da'].sel(mc_type='CHN').sum(dim=var_dim).to_pandas()


In [ ]:
# meta['mCGFrac'] = global_cg['mc'] / global_cg['cov']
# meta['mCHFrac'] = global_ch['mc'] / global_ch['cov']


In [ ]:
# meta.to_csv(f'{indir}MappingSummary.csv.gz')


In [ ]:
feature_cov_mean

In [ ]:
mc_type = 'CHN'
feature_cov_mean = mcds[f'{var_dim}_da'].sel(mc_type=mc_type, count_type='cov').mean(dim=obs_dim).to_pandas()
fig, ax = plt.subplots(figsize=(4,2), dpi=300)
sns.histplot(feature_cov_mean, bins=100, ax=ax)


In [ ]:
use_features = feature_cov_mean[feature_cov_mean > 1000].index
mcds = mcds.sel({var_dim:use_features})
mcds.add_mc_frac(var_dim=var_dim, normalize_per_cell=True, clip_norm_value=10)


In [ ]:
mch_adata = mcds.get_adata(mc_type=mc_type, var_dim=var_dim, select_hvf=False)


In [ ]:
model = PCA(n_components=100, svd_solver='arpack', random_state=0)
mch_adata.obsm['pca_all'] = model.fit_transform(mch_adata.X)
npc = significant_pc_test(mch_adata, obsm='pca_all', p_cutoff=0.01, update=False)


In [ ]:
npc = 50
mch_adata.obsm['100kCH_pca'] = mch_adata.obsm['pca_all'][:, :npc].copy()
tsne(mch_adata, obsm='100kCH_pca', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mch_adata.obsm[f'100kCH_pc{npc}_tsne'] = mch_adata.obsm['X_tsne'].copy()


In [ ]:
mch_adata = mch_adata[meta.index].copy()
mch_adata.obs = meta.copy()


In [ ]:
mch_adata

In [ ]:
# mch_adata.obsm['X_pca'] = normalize(mch_adata.obsm['100kCH_pca'][:,:10], axis=1)
# #mcad.obsm['X_pca'] = mcad.obsm['5kCG_pca'][:,:25]
# knn = 25
# sc.pp.neighbors(mch_adata, n_neighbors=knn, use_rep=f'X_pca')
# sc.tl.umap(mch_adata, maxiter=200, random_state=0)
# dump_embedding(mch_adata, 'umap')
# mch_adata.obsm[f'100kCH_pc10_umap'] = mch_adata.obsm['X_umap'].copy()


In [ ]:
L1_meta = pd.read_csv(f'{indir}L1color.tsv', sep='\t', header=0, index_col=0)
L1_annot = L1_meta['L1_abbr'].to_dict()
L1_color = L1_meta['color'].to_dict()
# L1_annot['c35'] = 'Hema Bnaive'
# L1_annot['c7'] = 'Hema Bmem'
# L1_color['c35'] = L1_color['c7']


In [ ]:
# L1_meta = pd.read_csv(f'{indir}L1color.tsv', sep='\t', header=0, index_col=0)
# L1_color = L1_meta['color'].to_dict()
tissue_meta = pd.read_csv(f'{indir}tissuecolor.tsv', sep='\t', header=0, index_col=0)
tissue_color = tissue_meta['color'].to_dict()


In [ ]:
mch_adata = anndata.read_h5ad(f'{outdir}cell_{meta.shape[0]}_100kCH.h5ad')


In [ ]:
knn = 25
sc.pp.neighbors(mch_adata, n_neighbors=knn, use_rep='100kCH_pca')
sc.tl.leiden(mch_adata, resolution=1.0, random_state=0, flavor='igraph')


In [ ]:
ds = 0.2
npc = 50
coord_base = 'tsne'
fig, ax = plt.subplots(figsize=(5, 4), dpi=300, constrained_layout=True)

tmp = mch_adata.obs.copy()
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='leiden',
                        text_anno='leiden',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )



In [ ]:
mch_adata.obs = meta.loc[mch_adata.obs.index].copy()

In [ ]:
mpl.use('agg')

In [ ]:
ds = 0.2
npc = 50
coord_base = 'tsne'
mch_adata.obsm[f'X_{coord_base}'] = mch_adata.obsm[f'100kCH_pc{npc}_{coord_base}'].copy()
dump_embedding(mch_adata, coord_base)

fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=300, constrained_layout=True)

tmp = mch_adata.obs.copy()
ax = axes[0, 0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # show_legend=True
                       )

ax = axes[0, 1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Tissue',
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette=tissue_color,
                        scatter_kws={'rasterized':True},
                        # show_legend=True,
                        # legend_kws={'ncol':1},
                       )

ax = axes[0, 2]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='L1',
                        text_anno=tmp['L1'].map(L1_annot), 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette=L1_color,
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )


ax = axes[1, 0]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue=np.log10(tmp['FinalmCReads']+1),
                       scatter_kws={'rasterized':True},
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCGFrac',
                       scatter_kws={'rasterized':True},
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       s=ds
                  )

ax = axes[1, 2]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCHFrac',
                       scatter_kws={'rasterized':True},
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       hue_norm=[0.005,0.02],
                       s=ds
                  )

fig.savefig(f'{outdir}All_100kCH_meta.pdf', transparent=True)


In [ ]:
ds = 0.2
npc = 50
coord_base = 'tsne'
# mch_adata.obsm[f'X_{coord_base}'] = mch_adata.obsm[f'100kCH_pc{npc}_{coord_base}'].copy()
# dump_embedding(mch_adata, coord_base)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=300, constrained_layout=True)

tmp = mch_adata.obs.copy()
ax = axes[0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='L1',
                        # text_anno=tmp['L1'].map(L1_annot), 
                        s=ds,
                        labelsize=8,
                        max_points=None,
                        palette=L1_color,
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1}, 
                        # show_legend=True
                       )

for xx,yy in L1_annot.items():
    x, y = np.mean(mch_adata.obsm[f'X_{coord_base}'][tmp['L1']==xx], axis=0)
    ax.text(x, y, yy, ha='center', va='center', fontsize=8)


ax = axes[1]
_ = continuous_scatter(ax=ax,
                       data=tmp,
                       hue='mCHFrac',
                       scatter_kws={'rasterized':True},
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=8,
                       cmap='vlag',
                       hue_norm=[0.005,0.02],
                       s=ds
                  )

fig.savefig(f'{outdir}All_100kCH_meta.pdf', transparent=True)


In [ ]:
mch_adata.write_h5ad(f'{outdir}cell_{mch_adata.shape[0]}_100kCH.h5ad')


In [ ]:
mch_adata = anndata.read_h5ad(f'{outdir}cell_86689_100kCH.h5ad')


In [ ]:
clf = LogisticRegression(class_weight='balanced')
skf = StratifiedKFold(n_splits=3, shuffle=True)
label = mch_adata.obs['L1'].copy()
leg = pd.Index(np.sort(label.unique()))
count = label.astype(str).value_counts()
score = np.zeros((2, len(leg), len(leg)))
data = mch_adata.obsm['100kCH_pca'].copy()
for i in np.arange(len(leg)-1):
    for j in np.arange(i+1, len(leg)):
        selc = label.isin([leg[i], leg[j]])
        if count.loc[leg[i]]<count.loc[leg[j]]:
            label_dict = {leg[i]:1, leg[j]:0}
        else:
            label_dict = {leg[i]:0, leg[j]:1}
        # selc = np.random.permutation(np.where(selc)[0])
        X = data[selc]
        y = label.values[selc].map(label_dict)
        pred = np.zeros(len(y))
        for train_index, test_index in skf.split(X, y):
            X_train, X_test, y_train, y_test = X[train_index], X[test_index], y[train_index], y[test_index]
            pred[test_index] = clf.fit(X_train, y_train).predict_proba(X_test)[:,1]
        score[0,i,j] = roc_auc_score(y, pred)
        score[1,i,j] = average_precision_score(y, pred)
        

In [ ]:
legname = leg.map(L1_annot)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6), dpi=300)
for i in range(2):
    tmp = pd.DataFrame(score[i], index=legname, columns=legname)
    tmp = tmp + tmp.T
    cg = sns.clustermap(tmp, metric='cosine', figsize=(0.1, 0.1))
    rorder = cg.dendrogram_row.reordered_ind.copy()
    corder = cg.dendrogram_col.reordered_ind.copy()
    ax = axes[i]
    ax.imshow(tmp.values[rorder][:, corder], cmap='cividis', vmin=[0.8, 0.6][i], vmax=1)
    ax.set_xticks(np.arange(len(leg)))
    ax.set_yticks(np.arange(len(leg)))
    ax.set_xticklabels(legname[corder], rotation=90)
    ax.set_yticklabels(legname[rorder])
    
fig.tight_layout()
fig.savefig('mCH_clustering/100kCH_L1_rocpr.pdf', transparent=True)


In [ ]:
L1_meta = L1_meta.drop(['c35','c36'], axis=0)


In [ ]:
from gliderport.preset import notebook_snakemake

# group_list = list(tissue_meta.index)
group_list = list(L1_meta.index)
print(len(group_list))


In [ ]:
notebook_snakemake(
    work_dir=f'{outdir}L1/',
    notebook_dir=f'{outdir}L1/template/',
    groups=group_list,
    default_cpu=4,
    default_mem_gb=16,
    redo_prepare=True,
)

In [ ]:
mch_adata.obs['L1'] = mch_adata.obs['L1'].astype(str)
selc = (mch_adata.obs['L1']=='c35')
mch_adata.obs.loc[selc, 'L1'] = 'c7'


In [ ]:
for xx in group_list:
    tmp = mch_adata[mch_adata.obs['L1']==xx].copy()
    tmp.write_h5ad(f'{outdir}L1/{xx}/100kCH.h5ad')
    

In [ ]:
!snakemake --snakefile Snakefile -j --keep-going


In [ ]:
coord_base = 'tsne'
# mpl.use('agg')

fig, axes = plt.subplots(7, 5, figsize=(8,8.5), dpi=300, constrained_layout=True)
for i,xx in enumerate(L1_meta.index):
    l1_name = L1_meta.loc[xx, 'L1_abbr']
    adata_tmp = anndata.read_h5ad(f'{outdir}L1/{xx}/100kCH_embed.h5ad')
    adata_tmp.obs['celltype'] = meta['celltype_L2_both_abbr'].astype(str)
    dump_embedding(adata_tmp, coord_base)
    leg = adata_tmp.obs['celltype'].unique()
    if len(leg)<=10:
        palette = 'muted'
    else:
        palette = 'tab20'
    print(xx, len(leg))
    ds = 20/np.sqrt(adata_tmp.shape[0])
    ax = axes.flatten()[i]
    tmp = adata_tmp.obs.copy()
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='celltype',
                            s=ds,
                            max_points=None,
                            palette=palette,
                            scatter_kws={'rasterized':True},
                            axis_format='empty',
                           )
    for yy,(x,y) in tmp.groupby('celltype')[['tsne_0', 'tsne_1']].mean().iterrows():
        ax.text(x, y, yy, ha='center', va='center', fontsize=6)
        
    ax.set_title(l1_name, fontsize=8)

for ax in axes.flatten()[L1_meta.shape[0]:]:
    ax.axis('off')
    
fig.savefig(f'{outdir}L1_mCH_seurat_tsne_celltype.pdf', transparent=True)


In [ ]:
from gliderport.preset import notebook_snakemake

group_list = list(tissue_meta.index)
# group_list = list(L1_meta.index)
print(len(group_list))


In [ ]:
notebook_snakemake(
    work_dir=f'{outdir}tissue/',
    notebook_dir=f'{outdir}tissue/template/',
    groups=group_list,
    default_cpu=4,
    default_mem_gb=16,
    redo_prepare=True,
)

In [ ]:
for xx in group_list:
    tmp = mch_adata[mch_adata.obs['Tissue']==xx].copy()
    tmp.write_h5ad(f'{outdir}{xx}/100kCH.h5ad')
    

In [ ]:
!snakemake --snakefile Snakefile -j --keep-going


In [ ]:
ds = 0.5
coord_base = 'tsne'
# mpl.use('agg')

fig, axes = plt.subplots(6, 3, figsize=(7, 10), dpi=300, constrained_layout=True)
for i,xx in enumerate(tissue_meta.index):
    tissue_name = tissue_meta.loc[xx, 'tissue_annot']
    adata_tmp = anndata.read_h5ad(f'{outdir}tissue/{xx}/100kCH_embed.h5ad')
    adata_tmp.obs['celltype'] = meta['celltype_L2_both_abbr'].astype(str)
    ax = axes.flatten()[i]
    count = adata_tmp.obs['celltype'].value_counts()
    leg = count.index[count>=30]
    selc = adata_tmp.obs['celltype'].isin(leg)
    tmp = adata_tmp.obs.loc[selc].copy()
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='celltype',
                            # text_anno='celltype', 
                            s=ds,
                            labelsize=6,
                            max_points=None,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1}, 
                            # show_legend=True
                           )
    for yy in leg:
        x, y = np.mean(adata_tmp.obsm[f'X_{coord_base}'][adata_tmp.obs['celltype']==yy], axis=0)
        ax.text(x, y, yy, ha='center', va='center', fontsize=6)
        
    ax.set_title(tissue_name, fontsize=8)

for ax in axes.flatten()[tissue_meta.shape[0]:]:
    ax.axis('off')
    
fig.savefig(f'{outdir}tissue_mCH_seurat_tsne_celltype.pdf', transparent=True)
